In [3]:
import os
import pandas as pd
import numpy as np

def process_patient_data(excel_file_path):
    try:
        patient_data = pd.read_excel(excel_file_path)
        # 假设数据中包含‘复查前结果’列，否则这里需要调整
        patient_data = patient_data.iloc[:, :patient_data.columns.get_loc('复查前结果')+1]
        columns_to_drop = ['序号', '姓名', '科室', '病区', '床号', '样本号', '条码号', '项目名称', '复查前结果']
        patient_data.drop(columns=[col for col in columns_to_drop if col in patient_data.columns], inplace=True)

        patient_data_grouped = patient_data.groupby(['病人号', '性别', '年龄', '样本日期', '项目英文缩写'])
        patient_data_processed = patient_data_grouped.first().reset_index()
        pivot_data = patient_data_processed.pivot_table(index=['病人号', '性别', '年龄', '样本日期'],
                                                        columns='项目英文缩写',
                                                        values='检验结果',
                                                        aggfunc='first',
                                                        fill_value=np.nan)
        pivot_data.reset_index(inplace=True)
        pivot_data.columns.name = None

        # 确保仅考虑指定列
        columns_to_check = ['ALB', 'ALT', 'AST', 'GLO', 'WBC', 'TP']
        features_to_fill = [col for col in columns_to_check if col in pivot_data.columns]
        condition = pivot_data['MDBFX'].notna() & pivot_data[features_to_fill].isna().any(axis=1)
        rows_to_check = pivot_data[condition]

        # 填充逻辑
        def fill_na_with_nearest(row, df):
            same_patient_data = df[(df['病人号'] == row['病人号']) & (df['MDBFX'].notna())].copy()
            same_patient_data['日期差异'] = abs(pd.to_datetime(same_patient_data['样本日期']) - pd.to_datetime(row['样本日期']))
            if len(same_patient_data) > 1:
                closest_row = same_patient_data.sort_values('日期差异').iloc[1]
                for feature in features_to_fill:
                    if pd.isna(row[feature]):
                        row[feature] = closest_row[feature]
            return row
        
        rows_to_check = rows_to_check.apply(lambda x: fill_na_with_nearest(x, pivot_data), axis=1)
        pivot_data.update(rows_to_check)

        # 删除 'MDBFX' 为空的行
        pivot_data = pivot_data.dropna(subset=['MDBFX'])

        return pivot_data

    except Exception as e:
        print(f"处理文件 {excel_file_path} 时发生错误：{e}")
        return pd.DataFrame()

def main():
    base_directory = "E:/Zhaolong/project/2404dianyong/raw_data/IMGT"
    years_and_folders = {
	    2020: ["2001.xls", "2002-2005.xls", "2006-2008.xls", "2009-2010.xls", "2011-2012.xls"],
	    2021:['2101-2102.xls','2103-2104.xls','2105-2106.xls','2107-2108.xls','2109-2110.xls','2111-2112.xls'],
	    2022:['2201-2202.xls','2203-2204.xls','2205-2206.xls','2207-2208.xls','2209-2210.xls','2211-2212.xls'],
	    2023:['2301-2302.xls','2303-2304.xls','2305-2306.xls','2307-2308.xls','2309-2310.xls','2311-2312.xls']
    }

    all_data = pd.DataFrame()

    for year, files in years_and_folders.items():
        for file_name in files:
            full_file_path = os.path.join(base_directory, str(year), file_name)
            file_data = process_patient_data(full_file_path)
            all_data = pd.concat([all_data, file_data], ignore_index=True)

    # 保存汇总数据到CSV
    output_csv_path = os.path.join(base_directory, "1_processed_patient_data.csv")
    all_data.to_csv(output_csv_path, encoding='gbk', index=False)  # 修改为'gbk'以解决乱码问题
    print(f"所有数据已处理完毕并保存到 {output_csv_path}")

if __name__ == "__main__":
    main()


WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** file size (3576693) not 512 + multiple of sector size (512)
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** file size (3455542) not 512 + multiple of sector size (512)
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** file size (3828916) not 512 + multiple of sector size (512)
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** file size (4124525) not 512 + multiple of sector size (512)
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT